In [1]:
import os
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from utils8 import AudioCNN
from utils8.data import AudioDataset, get_dataloader

# Get Data
path = os.path.join('Data', 'Digits')
classes = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
dataset = AudioDataset(path, classes)

# Data Split and Subsets
idx = list(range(len(dataset)))
labels = dataset.labels
train_val_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)

train_val_set = Subset(dataset, train_val_idx)
test_set = Subset(dataset, test_idx)

# Training

In [ ]:
import os
from sklearn.model_selection import KFold
from torch.utils.tensorboard import SummaryWriter
from torch import nn,optim
from utils8.dir_managment import clean_dir
from utils8.AudioCNN import AudioCNN
from utils8.training import train_one_fold, get_train_loaders, save_model_dict, save_target_labels


model_name = 'model_1'

log_dir = os.path.join('runs', 'training', model_name)
clean_dir(log_dir)
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

save_model_dir = os.path.join('Models', model_name)
clean_dir(save_model_dir)
os.makedirs(save_model_dir, exist_ok=True)

model_params = {'dropout_rate': 0.3, 'num_classes': 10}
save_model_dict(model_params, save_model_dir)
save_target_labels(classes, save_model_dir)

N_EPOCHS = 50
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_set)):
    print(f'\n ===================== Training Fold {fold}  ===================== \n')
    # Loaders
    train_loader, val_loader = get_train_loaders(train_val_set, train_idx,val_idx , batch_size=32)

    # Model, Optimizer, Criterion
    criterion = nn.CrossEntropyLoss()
    model = AudioCNN()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

    # Fold Training
    loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=save_model_dir, writer=writer)




Cleaning existing files at runs/training/model_1...
Cleaning existing files at Models/model_1...
Model dict saved in Models/model_1/model_dict.json
Labels dict saved in Models/model_1/data_labels.json

 ===================== Training Fold 0  ===================== 

Saved new best model to Models/model_1/fold_0.pth, with new best_val_loss=2.303948005040487
Saved new best model to Models/model_1/fold_0.pth, with new best_val_loss=2.3034232589933605


In [ ]:
# tensorboard --logdir Lab8/runs/training/model_1

# Model Evaluation

In [ ]:
import os
from utils8.Predictor import Predictor

model_path = os.path.join('Models', 'model_1')
predictor_obj = Predictor(model_path)

In [ ]:
model = predictor_obj.folds[0]

model

In [ ]:


loader = get_dataloader(dataset, batch_size=100)
x,y  =  next(iter(loader))

y

In [ ]:
from utils8.AudioCNN import AudioCNN
import torch

model1 = AudioCNN()

state_d = torch.load('Models/model_1/fold_1.pth', weights_only=True)
model1.load_state_dict(state_d)
model1(x).argmax(axis=1)

In [ ]:
# Train data Classification report
predictor_obj.metric_report(data=Subset(dataset, train_val_idx))


In [ ]:
# Test data Classification report
predictor_obj.metric_report(data=Subset(dataset, test_idx))